# 🩺 第三十天 · CBLUE 第二个子任务：CHIP-CTC（临床试验筛选标准分类）

**今日目标（约 1.5 小时）**：把 D26–D29 学到的「TF-IDF + LinearSVC」原样迁移到更难的任务——**CHIP-CTC**，44 类临床试验筛选标准分类。

**为什么是它**：临床试验「筛选标准」分类，正是**医学数据标注 / 临床研究 / 药企产品岗**的日常——"这条入选标准属于哪一类（如凝血功能、吸烟史、器官移植史）"。跑通它 = 简历上第二个对口任务。

**和 KUAKE-QIC 的区别**：11 类 → **44 类**；数据量 8.9 千 → **4 万条**；评估指标 accuracy → **Macro F1**（今天的新概念）。

## 第 0 步 · 认识 CHIP-CTC

| 项 | 值 |
|---|---|
| 任务 | 给一条临床试验「筛选标准」句子，判断属于 44 类里的哪一类 |
| 类别 | 44 类（含凝血功能、吸烟史、器官移植史、治疗方案…，见附件 category.xlsx） |
| 数据量 | 训练 22,962 / 验证 7,682 / 测试 10,000 |
| 标签 | **单标签**（每条句子只属 1 类，字段名是 `text` + `label`） |
| 指标 | **Macro F1**（不是准确率！） |

数据样例：
```json
{"id":"s2","label":"Addictive Behavior","text":"（2）重度吸烟（大于10支/天）及酗酒患者"}
```

## 第 1 步 · 下载数据（和 KUAKE-QIC 同一个地方）

1. 打开天池 CBLUE 数据集页：https://tianchi.aliyun.com/dataset/95414 （搜索「CBLUE」也能到）；
2. 下载 **CHIP-CTC.zip**；
3. 解压，把 `CHIP-CTC` 文件夹放到 `week5/` 下（和 `KUAKE-QIC` 并列）；
4. 里面应含 `CHIP-CTC_train.json` / `_dev.json` / `_test.json` + `category.xlsx`。

> 数据较大（4 万条），TF-IDF 训练会慢一点（几分钟），正常，耐心等。

In [1]:
import pandas as pd

train = pd.read_json("CHIP-CTC/CHIP-CTC_train.json")
dev   = pd.read_json("CHIP-CTC/CHIP-CTC_dev.json")

print("形状:", train.shape, dev.shape)
print("字段:", list(train.columns))
print()
print("44 类标签分布（训练集，按数量排序）:")
print(train["label"].value_counts())
print()
print("标签种类数:", train["label"].nunique())

形状: (22962, 3) (7682, 3)
字段: ['id', 'label', 'text']

44 类标签分布（训练集，按数量排序）:
label
Disease                             5127
Multiple                            4556
Therapy or Surgery                  1504
Consent                             1319
Diagnostic                          1233
Laboratory Examinations             1142
Pregnancy-related Activity          1026
Age                                  917
Pharmaceutical Substance or Drug     877
Risk Assessment                      708
Allergy Intolerance                  668
Enrollment in other studies          514
Researcher Decision                  464
Compliance with Protocol             370
Organ or Tissue Status               358
Sign                                 286
Addictive Behavior                   272
Capacity                             168
Life Expectancy                      166
Symptom                              154
Neoplasm Status                      131
Device                               129
Special Patient C

## 第 2 步 · 新概念：Macro F1 为什么替代准确率（Feynman 讲回给我）

**准确率在 44 类极不均衡时会骗人**：假设某类占了 40%，你「全猜这一类」也能拿 40% 准确率——看着还行，其实一个稀有类都没猜对。

**Macro F1 的做法**：
1. 对 **44 个类分别**算一个 F1（每个类单独看 precision/recall）；
2. 把 44 个 F1 **取平均**——**不看每类有多少条，一类一票**。

这样模型必须「连最小的类也要做好」，否则小类的低 F1 会把平均拉下去。

**类比**：一场考试 44 个考点，Macro F1 = 每个考点单独打分再平均；准确率 = 只看总分，你主攻最肥的考点就能刷分。医疗场景要的是**每类都靠谱**，所以用 Macro F1。

**对比记忆**：KUAKE-QIC 11 类较均衡 → 用 accuracy；CHIP-CTC 44 类极不均衡 → 用 Macro F1。

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

vec = TfidfVectorizer(analyzer="char", ngram_range=(1,2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train["text"])
Xde = vec.transform(dev["text"])

clf = LinearSVC(max_iter=5000)          # 44 类，迭代上限调大
clf.fit(Xtr, train["label"])
pred = clf.predict(Xde)

print("Macro F1 : %.4f" % f1_score(dev["label"], pred, average="macro"))
print("准确率   : %.4f" % accuracy_score(dev["label"], pred))

Macro F1 : 0.7779
准确率   : 0.8274


## 第 3 步 · 错误分析（写这里）

看两个分数：
- **Macro F1**：44 类每类一票的平均 F1——这才是 CHIP-CTC 官方指标；
- **准确率**：供参考，大概率比 Macro F1 高（大类拉高）。

**你的观察**（写这里）：
1. 哪个分数更高？为什么？  准确率高，因为44类数据的分布非常不均匀
2. 44 类里，猜哪些类最难？（提示：看 value_counts 里最少的几类）数量最少的那几类

## ✅ D30 完成标准

- [ ] 数据下载解压到 week5/CHIP-CTC/
- [ ] 预览跑通：44 类、字段、标签分布看清
- [ ] 新概念 Macro F1 能讲回给我
- [ ] baseline 跑通，Macro F1 分数出来
- [ ] 错误分析写了观察
- [ ] 保存（Cmd + S）

> 完成后喊我验收。今天最大的收获是「**迁移**」——同一套技能，换一个更难的真实任务，照样跑通。